
# 10 — Parallel AI Workflows: Concurrent Code Review

**Learning reference:** [BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Inspired by course notebook:** `parallelization.ipynb`

## Goal

Understand parallelization as an AI workflow architecture
by running independent tasks concurrently and combining
their results.

This notebook demonstrates:

- Sequential versus concurrent execution.
- Python's `asyncio`, `async` and `await`.
- Using `asyncio.gather()` for independent tasks.
- Collecting results from three different review roles.
- Running an optional live parallel workflow with Gemini.
- Handling individual request failures without automatic retries.

**Adaptation:** The course example analyzes a decision from
multiple perspectives and uses another LLM to aggregate
the answers. Here, three roles review a small Python
function, and Python combines the results locally.

**API usage:** The offline section makes zero requests.
The optional live section starts at most three Gemini
requests per initial execution. No separate aggregation
request is required.

The example source code is synthetic and is not executed.



## 1. Sequential vs. Parallel Execution

**Sequential execution** means completing one task
before starting the next.

Task A → Task B → Task C

**Parallelization** means independent tasks can make
progress during the same period.

Tasks A, B and C → Combined results

In Python, `asyncio` helps coordinate asynchronous tasks.

Key terms:

- `async def`: Defines an asynchronous function.
- `await`: Waits for an asynchronous operation to finish.
- `asyncio.gather()`: Runs multiple awaitable operations
  concurrently and collects their results.

This is especially useful for network requests, where
much of the time is spent waiting for a response.

Concurrency is not necessarily the same as running
Python calculations on multiple CPU cores.


In [1]:

import asyncio
import time

In [5]:

async def simulated_review(role):
    await asyncio.sleep(0.3)

    return {
        "role": role,
        "status": "completed"
    }


ROLES = [
    "Correctness",
    "Security",
    "Maintainability"
]

print("Simulated reviewer roles:", ROLES)


Simulated reviewer roles: ['Correctness', 'Security', 'Maintainability']


In [6]:

start = time.perf_counter()

sequential_results = []

for role in ROLES:
    result = await simulated_review(role)
    sequential_results.append(result)

sequential_seconds = time.perf_counter() - start

print("Sequential results:", sequential_results)
print(f"Elapsed time: {sequential_seconds:.3f} seconds")


Sequential results: [{'role': 'Correctness', 'status': 'completed'}, {'role': 'Security', 'status': 'completed'}, {'role': 'Maintainability', 'status': 'completed'}]
Elapsed time: 0.902 seconds


In [7]:

start = time.perf_counter()

parallel_results = await asyncio.gather(
    *(simulated_review(role) for role in ROLES)
)

parallel_seconds = time.perf_counter() - start

print("Parallel results:", parallel_results)
print(f"Elapsed time: {parallel_seconds:.3f} seconds")


Parallel results: [{'role': 'Correctness', 'status': 'completed'}, {'role': 'Security', 'status': 'completed'}, {'role': 'Maintainability', 'status': 'completed'}]
Elapsed time: 0.301 seconds


In [8]:

assert len(sequential_results) == 3
assert len(parallel_results) == 3

assert [r["role"] for r in sequential_results] == ROLES
assert [r["role"] for r in parallel_results] == ROLES

assert all(
    result["status"] == "completed"
    for result in parallel_results
)

print(f"Sequential time: {sequential_seconds:.3f}s")
print(f"Concurrent time: {parallel_seconds:.3f}s")

if parallel_seconds > 0:
    print(
        f"Observed speedup: "
        f"{sequential_seconds / parallel_seconds:.2f}x"
    )

print("\nAll offline concurrency checks passed.")
print("Gemini API requests: 0")


Sequential time: 0.902s
Concurrent time: 0.301s
Observed speedup: 2.99x

All offline concurrency checks passed.
Gemini API requests: 0



## 2. Three Independent Code-Review Perspectives

We will review a synthetic Python function that
retrieves a customer from a SQLite database.

Three reviewers examine the same function:

**Correctness Reviewer**
- Does the code handle valid and missing results?
- Can it raise unexpected errors?

**Security Reviewer**
- Are external inputs handled safely?
- Could the SQL query be manipulated?

**Maintainability Reviewer**
- Is the implementation clear and efficient?
- Can it be simplified or made easier to test?

These questions are related, but each review has
a different primary responsibility.

The sample is intentionally flawed so we can
compare the findings of independent reviewers.


In [9]:

CODE_TO_REVIEW = '''
def find_customer(conn, email):
    query = f"SELECT id, email FROM customers WHERE email = '{email}'"
    rows = conn.execute(query).fetchall()
    return rows[0]
'''.strip()

print("CODE TO REVIEW:\n")
print(CODE_TO_REVIEW)


CODE TO REVIEW:

def find_customer(conn, email):
    query = f"SELECT id, email FROM customers WHERE email = '{email}'"
    rows = conn.execute(query).fetchall()
    return rows[0]


In [10]:

def offline_review(role):
    findings = {
        "Correctness": (
            "rows[0] raises IndexError when no matching "
            "customer is returned."
        ),
        "Security": (
            "The SQL query interpolates email directly, "
            "creating a SQL injection risk."
        ),
        "Maintainability": (
            "fetchall() retrieves all matching rows "
            "although only the first result is used."
        )
    }

    return {
        "role": role,
        "finding": findings[role]
    }


offline_findings = [
    offline_review(role)
    for role in ROLES
]

for finding in offline_findings:
    print(f"\n{finding['role']}:")
    print(finding["finding"])

assert len(offline_findings) == 3
print("\nOffline review summaries prepared.")



Correctness:
rows[0] raises IndexError when no matching customer is returned.

Security:
The SQL query interpolates email directly, creating a SQL injection risk.

Maintainability:
fetchall() retrieves all matching rows although only the first result is used.

Offline review summaries prepared.



## 3. Live Parallel Code Review with Gemini

The live workflow sends the same Python function
to three model roles concurrently:

1. Correctness
2. Security
3. Maintainability

Each reviewer receives a different instruction,
so the analyses focus on separate concerns.

Python uses `asyncio.gather()` to coordinate
the three independent requests.

After the requests finish, their findings are
combined locally without an additional LLM call.

### API and Reliability Controls

- Three model requests in the initial live run.
- No automatic retries.
- Each role is stored separately after completion.
- A failed role does not erase successful role results.
- A rerun within the same Colab runtime skips roles
  that have already completed successfully.
- No external SQL queries or generated code are executed.

Three concurrent requests still consume three
requests from the applicable API quota.


In [11]:

from openai import AsyncOpenAI
from getpass import getpass

MODEL = "gemini-3.5-flash-lite"

gemini_key = getpass("Enter your Gemini API key: ")

async_client = AsyncOpenAI(
    api_key=gemini_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    max_retries=0,
    timeout=120.0
)

print("Selected model:", MODEL)
print("Async client ready.")
print("No API requests made.")


Enter your Gemini API key: ··········
Selected model: gemini-3.5-flash-lite
Async client ready.
No API requests made.


In [12]:

REVIEWER_INSTRUCTIONS = {
    "Correctness": (
        "You are a Python correctness reviewer. "
        "Identify functional bugs, edge cases, and "
        "possible runtime errors. "
        "Focus only on correctness."
    ),
    "Security": (
        "You are a Python security reviewer. "
        "Identify input-handling risks, injection "
        "vulnerabilities, and unsafe assumptions. "
        "Focus only on security."
    ),
    "Maintainability": (
        "You are a Python maintainability reviewer. "
        "Identify unnecessary complexity, inefficient "
        "patterns, and improvements to readability "
        "or testability. "
        "Focus only on maintainability."
    )
}


async def gemini_review(role):
    response = await async_client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": REVIEWER_INSTRUCTIONS[role]
            },
            {
                "role": "user",
                "content": (
                    "Review the following Python code.\n\n"
                    f"{CODE_TO_REVIEW}\n\n"
                    "Provide one or two specific findings. "
                    "Explain why they matter. "
                    "Be concise. "
                    "Do not execute or rewrite the code."
                )
            }
        ],
        temperature=0.2
    )

    content = response.choices[0].message.content

    if not content:
        raise ValueError(
            f"{role} reviewer returned empty content."
        )

    return {
        "role": role,
        "analysis": content
    }


print("Three Gemini reviewer roles defined.")
print("No API requests made.")


Three Gemini reviewer roles defined.
No API requests made.


In [13]:


if "live_results" not in globals():
    live_results = {}

if "live_errors" not in globals():
    live_errors = {}


async def run_pending_reviews():
    pending = [
        role
        for role in ROLES
        if role not in live_results
    ]

    if not pending:
        print("All reviews already completed.")
        print("No new API requests needed.")
        return

    print("Starting reviews:", pending)
    print("API requests to attempt:", len(pending))

    started = time.perf_counter()

    outcomes = await asyncio.gather(
        *(gemini_review(role) for role in pending),
        return_exceptions=True
    )

    elapsed = time.perf_counter() - started

    for role, outcome in zip(pending, outcomes):
        if isinstance(outcome, BaseException):
            live_errors[role] = (
                f"{type(outcome).__name__}: {outcome}"
            )
            print(
                f"\n{role} review failed:",
                live_errors[role]
            )
        else:
            live_results[role] = outcome
            live_errors.pop(role, None)
            print(f"\n{role} review completed.")

    print(f"\nElapsed time: {elapsed:.2f} seconds")
    print("Successful roles:", list(live_results))
    print("Failed roles:", list(live_errors))


await run_pending_reviews()


Starting reviews: ['Correctness', 'Security', 'Maintainability']
API requests to attempt: 3

Correctness review failed: InternalServerError: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]

Security review completed.

Maintainability review completed.

Elapsed time: 83.58 seconds
Successful roles: ['Security', 'Maintainability']
Failed roles: ['Correctness']


In [14]:

# Retry only the failed review.
# Previously successful results remain saved.

await run_pending_reviews()


Starting reviews: ['Correctness']
API requests to attempt: 1

Correctness review completed.

Elapsed time: 68.95 seconds
Successful roles: ['Security', 'Maintainability', 'Correctness']
Failed roles: []


In [15]:

print("PARALLEL CODE REVIEW REPORT")
print("=" * 50)

for role in ROLES:
    print(f"\n### {role}")

    if role in live_results:
        print(live_results[role]["analysis"])

    elif role in live_errors:
        print("Review unavailable.")
        print("Reason:", live_errors[role])

    else:
        print("Review not executed.")

print("\n" + "=" * 50)

print(
    f"Completed: {len(live_results)}/{len(ROLES)}"
)

if len(live_results) == len(ROLES):
    print("All three parallel reviews completed.")
else:
    print("Report is partial; some reviews are missing.")


PARALLEL CODE REVIEW REPORT

### Correctness
1. **SQL Injection Vulnerability**: The code uses Python f-strings to embed the `email` variable directly into the SQL query string. If a malicious user supplies input containing SQL commands (e.g., `' OR '1'='1`), it can manipulate the query logic, leading to unauthorized data access or database compromise. Parameterized queries should be used instead.

2. **IndexError on Empty Results**: The code directly returns `rows[0]` without checking if the query returned any results. If no customer matches the given email, `rows` will be an empty list, and accessing `rows[0]` will raise an unhandled `IndexError`, crashing the function.

### Security
**Finding 1: SQL Injection Vulnerability**
* **Risk:** The function uses Python f-strings to format the `email` parameter directly into the SQL query string. 
* **Why it matters:** An attacker can supply malicious input (e.g., `' OR '1'='1`) via the `email` argument to alter the structure of the query, p


## 4. What Parallelization Demonstrates

Parallel execution is useful when tasks are independent.

Here, the three reviewers can analyze the same Python
function without waiting for one another's findings.

The model roles share the same underlying Gemini model,
but receive different instructions.

### Benefits

- Independent requests can overlap in time.
- Each role concentrates on a specific review dimension.
- Results can be collected and organized by Python.
- Individual failures can be recorded separately.

### Trade-offs

- Parallelization does not reduce the number of API requests.
- Concurrent requests may encounter rate limits or server errors.
- Reviewers can produce overlapping or contradictory findings.
- The combined report does not independently verify accuracy.
- More complex tasks may have dependencies that prevent
  concurrent execution.

The notebook uses one parallel review round rather
than an autonomous multi-step agent loop.


In [16]:

completed_roles = [
    role
    for role in ROLES
    if role in live_results
]

missing_roles = [
    role
    for role in ROLES
    if role not in live_results
]

print("Model:", MODEL)
print("Expected reviewers:", ROLES)
print("Completed reviewers:", completed_roles)
print("Missing reviewers:", missing_roles)

print("\nSaved analyses:")

for role in completed_roles:
    analysis = live_results[role]["analysis"]

    print(
        f"- {role}: {len(analysis.split())} words"
    )

if not missing_roles:
    print("\nStructural verification: PASSED")
else:
    print("\nStructural verification: INCOMPLETE")

print("Additional Gemini API requests: 0")


Model: gemini-3.5-flash-lite
Expected reviewers: ['Correctness', 'Security', 'Maintainability']
Completed reviewers: ['Correctness', 'Security', 'Maintainability']
Missing reviewers: []

Saved analyses:
- Correctness: 95 words
- Security: 115 words
- Maintainability: 134 words

Structural verification: PASSED
Additional Gemini API requests: 0



## Findings and Limitations

### Offline Concurrency

- Compared sequential and concurrent execution of three
  simulated independent tasks.
- Sequential execution took 0.902 seconds.
- Concurrent execution took 0.301 seconds.
- The observed speedup was approximately 2.99×.
- All offline concurrency assertions passed.

### Live Parallel Code Review

- Used Gemini 3.5 Flash Lite for three independent reviews:
  correctness, security, and maintainability.
- Executed the initial requests concurrently using
  `asyncio.gather()` with `return_exceptions=True`.
- Security and Maintainability completed successfully.
- Correctness initially failed with HTTP 503.
- A separate targeted retry completed the missing
  Correctness review without repeating successful requests.
- Python combined all three reviews locally without
  an additional aggregation request.

### Observed Findings

- All three reviewers identified SQL injection risks caused
  by direct string interpolation in the SQL query.
- All three also identified the unhandled `IndexError`
  when no matching customer exists.
- The reviewers' findings overlapped despite their
  different instructions.
- The offline Maintainability review identified unnecessary
  use of `fetchall()`, while the live reviewer focused on
  security and robustness issues.

### API Usage

- Initial parallel execution: 3 attempted Gemini requests.
- Targeted retry: 1 additional attempted request.
- Local aggregation and verification: 0 additional requests.
- Total: 4 attempted Gemini requests.

### Limitations

- The reviewed source code is synthetic and was not executed.
- The offline reviews use predefined findings.
- All live reviewers use the same Gemini model with
  different role instructions.
- AI-generated findings were not independently validated
  through executable tests.
- Parallelization can reduce elapsed waiting time but
  does not reduce the number of API requests.
- The observed offline speedup is based on simulated waits
  and does not establish equivalent live API performance.
- Saved successful results are retained only within
  the active Colab runtime.

### Conclusion

This notebook demonstrates parallelization as a workflow
architecture for independent AI tasks.

The offline experiment shows how asynchronous execution
can overlap waiting periods and reduce total elapsed time.

The live experiment demonstrates concurrent model requests,
local result aggregation, and recovery from an individual
API failure without repeating successful work.

The results also show that role-specific prompts do not
guarantee independent or non-overlapping insights. Parallel
AI reviews should therefore be validated and interpreted
critically rather than treated as independent confirmation.
